<img src="https://drive.google.com/uc?export=view&id=1wYSMgJtARFdvTt5g7E20mE4NmwUFUuog" width="200">

[![Gen AI Experiments](https://img.shields.io/badge/Gen%20AI%20Experiments-GenAI%20Bootcamp-blue?style=for-the-badge&logo=artificial-intelligence)](https://github.com/buildfastwithai/gen-ai-experiments)
[![Gen AI Experiments GitHub](https://img.shields.io/github/stars/buildfastwithai/gen-ai-experiments?style=for-the-badge&logo=github&color=gold)](http://github.com/buildfastwithai/gen-ai-experiments)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1wy6wR9w9NwQkNg2a3QvDA0UcXEldsu4K?usp=sharing)






# Mercury 2 Cookbook

A comprehensive guide demonstrating all use cases and functionality of the **Mercury 2** LLM via the Inception API.

**Topics covered:**
1. Setup & Configuration  
2. Chat Completions  
3. Streaming  
4. Diffusing Mode  
5. Instant Mode (Low Latency)  
6. Tool Use (Function Calling)  
7. Structured Outputs  
8. Autocomplete (FIM) – Mercury Edit  
9. Next Edit – Mercury Edit

---

## 1. Setup & Configuration

Configure the OpenAI-compatible client to use the Inception API base URL. Load your API key from the `..env` file.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load API key from ..env (or .env)
load_dotenv("..env")
load_dotenv(".env")
api_key = os.getenv("INCEPTION_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.inceptionlabs.ai/v1"
)

print("Client configured successfully.")

Client configured successfully.


---

## 2. Chat Completions

Generate conversational chat completions with system and user messages.

In [2]:
response = client.chat.completions.create(
    model="mercury-2",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is a diffusion model?"}
    ],
    max_tokens=1000
)

print(response.choices[0].message.content)

A **diffusion model** is a type of generative machine‑learning model that creates new data (images, text, audio, etc.) by learning how to reverse a gradual “noising” process. The core idea can be broken down into two complementary phases:

---

## 1. Forward (diffusion) process  
- **Goal:** Transform a real data sample into pure noise in a series of small, stochastic steps.  
- **How it works:** Starting from a clean example \(x_0\) (e.g., an image), we add a tiny amount of Gaussian noise at each step \(t = 1, \dots, T\). After many steps the sample becomes indistinguishable from a standard normal distribution \( \mathcal{N}(0, I) \).  
- **Mathematics:**  
  \[
  q(x_t \mid x_{t-1}) = \mathcal{N}\bigl(x_t; \sqrt{1-\beta_t}\,x_{t-1}, \beta_t I\bigr)
  \]
  where \(\beta_t\) is a small variance schedule. The whole chain has a closed‑form marginal:
  \[
  q(x_t \mid x_0) = \mathcal{N}\bigl(x_t; \sqrt{\bar\alpha_t}\,x_0, (1-\bar\alpha_t)I\bigr)
  \]
  with \(\alpha_t = 1-\beta_t\) and \(

---

## 3. Streaming

Get responses block-by-block for instant feedback—ideal for chat and live applications.

In [3]:
stream = client.chat.completions.create(
    model="mercury-2",
    messages=[{"role": "user", "content": "Explain recursion in one paragraph."}],
    max_tokens=500,
    stream=True
)

full_content = ""
for chunk in stream:
    if chunk.choices[0].delta.content:
        full_content += chunk.choices[0].delta.content
        print(chunk.choices[0].delta.content, end="")

print("\n\n--- Full response length:", len(full_content), "chars")

Recursion is a programming technique in which a function solves a problem by calling itself on a smaller or simpler instance of that same problem, typically until it reaches a “base case” that can be solved directly without further self‑calls; each recursive call builds up a stack of pending operations that are resolved in reverse order as the base case is hit, allowing complex tasks—such as traversing hierarchical data structures, computing factorials, or performing divide‑and‑conquer algorithms—to be expressed elegantly and concisely.

--- Full response length: 542 chars


---

## 4. Diffusing Mode

Visualize how noisy outputs are refined into final text, showcasing the model's iterative denoising process. Use `stream=True` and `diffusing=True`.

In [4]:
import requests
import json

url = "https://api.inceptionlabs.ai/v1/chat/completions"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {api_key}"
}
payload = {
    "model": "mercury-2",
    "messages": [{"role": "user", "content": "What is a diffusion model?"}],
    "max_tokens": 500,
    "stream": True,
    "diffusing": True
}

resp = requests.post(url, headers=headers, json=payload, stream=True)
full_content = ""
for line in resp.iter_lines():
    if line:
        line = line.decode("utf-8")
        if line.startswith("data: ") and line != "data: [DONE]":
            try:
                data = json.loads(line[6:])
                for choice in data.get("choices", []):
                    delta = choice.get("delta", {})
                    if delta.get("content"):
                        full_content = delta["content"]  # In diffusing mode, full content is returned per chunk
            except json.JSONDecodeError:
                pass

print("Diffusing response:")
print(full_content)

Diffusing response:
A diffusion model is a type of generative model that learns to create data (e.g., images, audio, text) by gradually “denoising” a random signal. The core idea is to define a **two‑step process**:

1. **Forward (noising) process** – Starting from a real data sample, the model adds a small amount of Gaussian noise repeatedly over many discrete time steps \(t = 1 \dots T\). After enough steps the data becomes indistinguishable from pure Gaussian noise. This forward process is a fixed Markov chain whose transition probabilities are analytically known.

2. **Reverse (denoising) process** – The model is trained to approximate the inverse of the forward chain: given a noisy sample at step \(t\), it predicts a slightly less‑noisy version at step \(t-1\). By chaining these reverse steps together, starting from pure noise, the model can generate a new data sample.

### How it works in practice
- **Training objective**: Typically a simple mean‑squared error between the model’s

---

## 5. Instant Mode (Low Latency)

Use `reasoning_effort="instant"` for near-instant responses. Ideal for voice assistants, chatbots, real-time systems, and low-latency workflows.

In [5]:
response = client.chat.completions.create(
    model="mercury-2",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "What is 15 + 27?"}
    ],
    reasoning_effort="instant"
)

print(response.choices[0].message.content)

42


---

## 6. Tool Use (Function Calling)

Inception API supports tool calling for complex workflows. Define tools and let the model decide when to call them.

In [6]:
import json

tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City and state, e.g., San Francisco, CA"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
            },
            "required": ["location", "unit"]
        }
    }
}]

response = client.chat.completions.create(
    model="mercury-2",
    messages=[{"role": "user", "content": "What's the weather like in San Francisco?"}],
    tools=tools
)

msg = response.choices[0].message
if msg.tool_calls:
    tc = msg.tool_calls[0]
    fn = tc.function
    print(f"Function called: {fn.name}")
    print(f"Arguments: {fn.arguments}")
    args = json.loads(fn.arguments)
    # Simulate tool execution
    print(f"\n[Simulated] Getting weather for {args['location']} in {args['unit']}")
else:
    print(msg.content)

Function called: get_weather
Arguments: {
  "location": "San Francisco, CA",
  "unit": "fahrenheit"
}

[Simulated] Getting weather for San Francisco, CA in fahrenheit


---

## 7. Structured Outputs

Use `response_format` with a JSON schema to enforce structured responses with specific properties and types.

In [7]:
response_schema = {
    "name": "Sentiment",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]},
            "confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "key_phrases": {"type": "array", "items": {"type": "string"}}
        },
        "required": ["sentiment", "confidence", "key_phrases"]
    }
}

response = client.chat.completions.create(
    model="mercury-2",
    messages=[{
        "role": "user",
        "content": "Analyze the sentiment of this text: 'I absolutely love this feature! It works perfectly and saves me so much time.'"
    }],
    response_format={"type": "json_schema", "json_schema": response_schema},
    stream=False
)

import json
result = json.loads(response.choices[0].message.content)
print(json.dumps(result, indent=2))

{
  "sentiment": "positive",
  "confidence": 0.98,
  "key_phrases": [
    "absolutely love this feature",
    "works perfectly",
    "saves me so much time"
  ]
}


---

## 8. Autocomplete (FIM) – Mercury Edit

Fill-in-the-middle completions using the `mercury-edit` model. Provide `prompt` (before cursor) and `suffix` (after cursor).

In [8]:
import requests

url = "https://api.inceptionlabs.ai/v1/fim/completions"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {api_key}"
}
payload = {
    "model": "mercury-edit",
    "prompt": "def fibonacci(",
    "suffix": "return a + b"
}

resp = requests.post(url, headers=headers, json=payload)
data = resp.json()
choices = data.get("choices", [])
if choices:
    text = choices[0].get("text", "")
    print("FIM completion:")
    print(f"def fibonacci({text}return a + b")

FIM completion:
def fibonacci(a, b):
    if a > b:
        return a - b
    else:
        return a + b


---

## 9. Next Edit – Mercury Edit

Generate next-edit completions for code. The request must include recently viewed snippets, current file content with editable region (`<|code_to_edit|>`), and optional edit history.

In [9]:
import requests

url = "https://api.inceptionlabs.ai/v1/edit/completions"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {api_key}"
}

content = '''<|recently_viewed_code_snippets|>

<|/recently_viewed_code_snippets|>

<|current_file_content|>
current_file_path: solver.py
\"\"\"
function: flagAllNeighbors
----------
This function marks each of the covered neighbors of the cell at the given row and col as flagged.
\"\"\"
<|code_to_edit|>
def flagAllNeighbors(board<|cursor|>, row, col):
    for r, c in b.getNeighbors(row, col):
        if b.isValid(r, c):
            b.flag(r, c)
<|/code_to_edit|>
<|/current_file_content|>

<|edit_diff_history|>
--- /c:/Users/test/testing/solver.py
+++ /c:/Users/test/testing/solver.py
@@ -6,1 +6,1 @@
-def flagAllNeighbors(b, row, col):
+def flagAllNeighbors(board, row, col):
<|/edit_diff_history|>'''

payload = {
    "model": "mercury-edit",
    "messages": [{"role": "user", "content": content}]
}

resp = requests.post(url, headers=headers, json=payload)
data = resp.json()
choices = data.get("choices", [])
if choices:
    print("Next Edit suggestion:")
    print(choices[0].get("message", {}).get("content", ""))

Next Edit suggestion:
```
def flagAllNeighbors(board, row, col):
    for r, c in board.getNeighbors(row, col):
        if board.isValid(r, c):
            board.flag(r, c)
```
